In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import torch.nn as nn
from torchinfo import summary
import statistics
import csv

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
all = []
pina = []
betu = []
cupr = []
sapi = []
faga = []
with open('embedding_dataset.csv', 'r') as csvfile:
    reader = csv.reader(csvfile)
    next(reader) #skips first line of csv with headers
    for line in reader:
        match line[0]:
            case 'Pinaceae':
                pina.append(line)
                all.append(line)
            case 'Betulaceae':
                betu.append(line)
                all.append(line)
            case 'Cupressaceae':
                cupr.append(line)
                all.append(line)
            case 'Sapindaceae':
                sapi.append(line)
                all.append(line)
            case 'Fagaceae':
                faga.append(line)
                all.append(line)
            case _:
                print('unplanned classification')

In [4]:
length = len(all)
print(length)
print(len(pina))
print(len(betu))
print(len(cupr))
print(len(sapi))
print(len(faga))

49627
14230
2650
3833
8839
20075


In [ ]:
genus_toint_dict = {}
species_toint_dict = {}
family_toint_dict = {}
location_toint_dict = {}
genus_tostring_dict = {}
species_tostring_dict = {}
family_tostring_dict = {}
location_tostring_dict = {}

fencode = 0
gencode = 0
sencode = 0
lencode = 0

for entry in all:
    if not entry[0] in family_toint_dict:
        family_toint_dict[entry[0]] = fencode
        family_tostring_dict[fencode] = entry[0]
        fencode += 1
    if not entry[1] in genus_toint_dict:
        genus_toint_dict[entry[1]] = gencode
        genus_tostring_dict[gencode] = entry[1]
        gencode += 1
    if not entry[2] in species_toint_dict:
        species_toint_dict[entry[2]] = sencode
        species_tostring_dict[sencode] = entry[2]
        sencode += 1
    if not entry[4] in location_toint_dict:
        location_toint_dict[entry[4]] = lencode
        location_tostring_dict[lencode] = entry[4]
        lencode += 1


In [29]:
def encode(dataset):
    family = []
    genus = []
    species = []
    year = []
    embeddings_tensor = torch.zeros(len(dataset), 64)
    for i, entry in enumerate(dataset):
        family.append(family_toint_dict[entry[0]])
        genus.append(genus_toint_dict[entry[1]])
        species.append(species_toint_dict[entry[2]])
        year.append(int(entry[3]))
        embeddings_tensor[i] = torch.tensor(list(map(float, entry[5:])))

    family_tensor = torch.tensor(family)
    genus_tensor = torch.tensor(genus)
    species_tensor = torch.tensor(species)
    year_tensor = torch.tensor(year)

    return family_tensor, genus_tensor, species_tensor, year_tensor, embeddings_tensor

In [31]:
pina_family_tensor, pina_genus_tensor, pina_species_tensor, pina_year_tensor, pina_embeddings_tensor = encode(pina)
betu_family_tensor, betu_genus_tensor, betu_species_tensor, betu_year_tensor, betu_embeddings_tensor = encode(betu)
cupr_family_tensor, cupr_genus_tensor, cupr_species_tensor, cupr_year_tensor, cupr_embeddings_tensor = encode(cupr)
sapi_family_tensor, sapi_genus_tensor, sapi_species_tensor, sapi_year_tensor, sapi_embeddings_tensor = encode(sapi)
faga_family_tensor, faga_genus_tensor, faga_species_tensor, faga_year_tensor, faga_embeddings_tensor = encode(faga)
m_family_tensor, m_genus_tensor, m_species_tensor, m_year_tensor, m_embeddings_tensor = encode(all)
master = [m_family_tensor, m_genus_tensor, m_species_tensor, m_year_tensor, m_embeddings_tensor]

In [20]:
species_weight = np.array([0.0]*len(species_toint_dict))

for entry in master[2]:
    species_weight[entry] += 1.0

species_weight = torch.tensor(species_weight/length).to(device)

In [32]:
class embed_dataset(torch.utils.data.Dataset):
    def __init__(self, family, genus, species, year, embeddings):
        # self.family = family
        # self.genus = genus
        self.species = species
        # self.year = year
        self.embeddings = embeddings

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        # family = self.family[idx]
        # genus = self.genus[idx]

        species = self.species[idx]
        label = torch.zeros(len(species_weight))
        label[species] = 1
        target = species
        # year = self.year[idx]
        embedding = self.embeddings[idx]
        return label, embedding, target


# Create dataset splits
# Set up dataloader for batches
batch_size = 1

pina_train_dataset = embed_dataset(pina_family_tensor, pina_genus_tensor, pina_species_tensor, pina_year_tensor, pina_embeddings_tensor)
pina_train_dataloader = DataLoader(pina_train_dataset, batch_size=batch_size, shuffle=True)
print(len(pina_train_dataloader))
betu_train_dataset = embed_dataset(betu_family_tensor, betu_genus_tensor, betu_species_tensor, betu_year_tensor, betu_embeddings_tensor)
betu_train_dataloader = DataLoader(betu_train_dataset, batch_size=batch_size, shuffle=True)

cupr_train_dataset = embed_dataset(cupr_family_tensor, cupr_genus_tensor, cupr_species_tensor, cupr_year_tensor, cupr_embeddings_tensor)
cupr_train_dataloader = DataLoader(cupr_train_dataset, batch_size=batch_size, shuffle=True)

sapi_train_dataset = embed_dataset(sapi_family_tensor, sapi_genus_tensor, sapi_species_tensor, sapi_year_tensor, sapi_embeddings_tensor)
sapi_train_dataloader = DataLoader(sapi_train_dataset, batch_size=batch_size, shuffle=True)

faga_train_dataset = embed_dataset(faga_family_tensor, faga_genus_tensor, faga_species_tensor, faga_year_tensor, faga_embeddings_tensor)
faga_train_dataloader = DataLoader(faga_train_dataset, batch_size=batch_size, shuffle=True)

14230


In [33]:
#define model parameters
class SimpleLinear2(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(64, 1024)
        self.l2 = nn.Linear(1024, 2048)
        self.l3 = nn.Linear(2048, 242)
        self.relu = nn.ReLU()

    def forward(self, embedding):
        x = self.relu(self.l1(embedding))
        x = self.relu(self.l2(x))
        return self.l3(x)

In [34]:
import torch.optim as optim


def training(model, learning_rate, criterion, train_dataloader):
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

    for epoch in range(10):
        model.train()
        total_loss = 0
        for label, embedding, target in train_dataloader:
            embedding, label = embedding.to(device), label.to(device)
            optimizer.zero_grad()
            logits = model(embedding)
            loss = criterion(logits, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_loss = total_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} - Loss: {avg_loss:.6f}")

In [ ]:
import torch.optim as optim

learning_rate = 2e-3

pinalinear2 = SimpleLinear2().to(device)
betulinear2 = SimpleLinear2().to(device)
cuprlinear2 = SimpleLinear2().to(device)
sapilinear2 = SimpleLinear2().to(device)
fagalinear2 = SimpleLinear2().to(device)
# linear3 = SimpleLinear3().to(device)
# linear4 = SimpleLinear4().to(device)
# skip4 = SkipLinear4().to(device)

criterion = nn.BCEWithLogitsLoss(weight=species_weight)

training(model=pinalinear2, learning_rate=learning_rate, criterion=criterion, train_dataloader=pina_train_dataloader)
training(model=betulinear2, learning_rate=learning_rate, criterion=criterion, train_dataloader=betu_train_dataloader)
training(model=cuprlinear2, learning_rate=learning_rate, criterion=criterion, train_dataloader=cupr_train_dataloader)
training(model=sapilinear2, learning_rate=learning_rate, criterion=criterion, train_dataloader=sapi_train_dataloader)
training(model=fagalinear2, learning_rate=learning_rate, criterion=criterion, train_dataloader=faga_train_dataloader)
# training(model=linear3, learning_rate=learning_rate, criterion=criterion)
# training(model=linear4, learning_rate=learning_rate, criterion=criterion)
# training(model=skip4, learning_rate=learning_rate, criterion=criterion)

In [45]:
def analyze(model, learning_rate, dataloader):
    count = 0
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate)
    with torch.no_grad():
        for label, embedding, target in dataloader:
            embedding, label, target = embedding.to(device), label.to(device), target.to(device)
            optimizer.zero_grad()
            logits = model(embedding)
            loss = criterion(logits, label)
            predict = nn.Softmax()
            prediction = predict(logits)
            if(torch.argmax(prediction) == target):
                count += 1
    print(count, len(dataloader))
    print(count/len(dataloader))

def run_tests(model):
    analyze(model=model, learning_rate=learning_rate, dataloader=pina_train_dataloader)
    analyze(model=model, learning_rate=learning_rate, dataloader=betu_train_dataloader)
    analyze(model=model, learning_rate=learning_rate, dataloader=cupr_train_dataloader)
    analyze(model=model, learning_rate=learning_rate, dataloader=sapi_train_dataloader)
    analyze(model=model, learning_rate=learning_rate, dataloader=faga_train_dataloader)

In [46]:
run_tests(model=pinalinear2)
run_tests(model=betulinear2)
run_tests(model=cuprlinear2)
run_tests(model=sapilinear2)
run_tests(model=fagalinear2)

/home/april/final_deep/lib/python3.12/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


11094 14230
0.7796205200281097
0 2650
0.0
0 3833
0.0
0 8839
0.0
0 20075
0.0
0 14230
0.0
2339 2650
0.8826415094339622
0 3833
0.0
0 8839
0.0
0 20075
0.0
0 14230
0.0
0 2650
0.0
3691 3833
0.9629533002869815
0 8839
0.0
0 20075
0.0
0 14230
0.0
0 2650
0.0
0 3833
0.0
5636 8839
0.6376286910283969
0 20075
0.0
0 14230
0.0
0 2650
0.0
0 3833
0.0
0 8839
0.0
11242 20075
0.56


In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, layers=1):
        super().__init__()
        self.t1 = nn.TransformerEncoderLayer(d_model=64, nhead = 8, dim_feedforward=2048, dropout=0.1, batch_first=True)
        self.t2 = nn.TransformerEncoderLayer(d_model=64, nhead = 8, dim_feedforward=2048, dropout=0.1, batch_first=True)
        self.t3 = nn.TransformerEncoderLayer(d_model=64, nhead = 8, dim_feedforward=2048, dropout=0.1, batch_first=True)
        self.l1 = nn.Linear(64, 1024)
        self.l2 = nn.Linear(1024, 242)
        self.relu = nn.ReLU()
        self.layers = layers
    def forward(self, embedding):
        result = self.t3(self.t2(self.t1(embedding)))
        x = self.relu(self.l1(result))
        return self.l2(x)

In [ ]:
learning_rate = 2e-4
transformer1 = TransformerModel(layers=4).to(device)

criterion = nn.BCEWithLogitsLoss(weight=species_weight)

training(model=transformer1, learning_rate=learning_rate, criterion=criterion)

Epoch 1 - Loss: 0.000136
Epoch 2 - Loss: 0.000118
Epoch 3 - Loss: 0.000113
Epoch 4 - Loss: 0.000110
Epoch 5 - Loss: 0.000108
Epoch 6 - Loss: 0.000107
Epoch 7 - Loss: 0.000105
Epoch 8 - Loss: 0.000104
Epoch 9 - Loss: 0.000104
Epoch 10 - Loss: 0.000103


In [ ]:
analyze(transformer1, learning_rate)

/home/april/final_deep/lib/python3.12/site-packages/torch/nn/modules/module.py:1779: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


24472 49627
0.49311866524271064
